# Preprocessing ML Utilities Demo

Demostración interactiva de las funciones de preparación integral de DataFrames para algoritmos de Machine Learning implementadas en `_preprocessing_ml.py` y expuestas por `ds_utils.preprocessing.preprocessing_utilities`.

Funciones cubiertas en este notebook:
- `prepare_dataframe_for_ml`

## Imports

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from ds_utils.preprocessing.preprocessing_utilities import (
    prepare_dataframe_for_ml,
)

## prepare_dataframe_for_ml

`prepare_dataframe_for_ml` adecúa un conjunto de datos a formatos listos para entrenamiento de modelos de Machine Learning:
- Detecta y convierte cadenas de texto booleanas (`'true'`, `'false'`) a booleanos o enteros binarios (`bool_as_int=True`).
- Convierte cadenas numéricas a tipos numéricos nativos.
- Transforma marcas de tiempo (`datetime64`) en timestamps enteros (`int64`).
- Procesa variables categóricas según la estrategia deseada (`'category'`, `'onehot'`, `'label'`, `'drop'`).
- Rellena nulos con `0` en columnas numéricas especificadas (`fillna_cols`).
- Excluye identificadores o variables objetivo de cualquier transformación (`excluded_cols`).

In [2]:
# 1. DataFrame heterogéneo con cadenas numéricas, booleanas, fechas y nulos
df_raw_ml = pd.DataFrame({
    "target_id": ["ID_1", "ID_2", "ID_3", "ID_4"],
    "str_bool": ["true", "false", "TRUE", "false"],
    "str_numeric": ["10.5", "20.0", "30.2", "40.8"],
    "category_col": ["baja", "media", "alta", "media"],
    "event_date": pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04"]),
    "numeric_nan": [100.0, np.nan, 300.0, np.nan]
})

print("DataFrame original:")
display(df_raw_ml)
print("Tipos de datos originales:")
print(df_raw_ml.dtypes)

DataFrame original:


,target_id,str_bool,str_numeric,category_col,event_date,numeric_nan
0,ID_1,true,10.5,baja,2024-01-01,100.0
1,ID_2,false,20.0,media,2024-01-02,NaN
2,ID_3,TRUE,30.2,alta,2024-01-03,300.0
3,ID_4,false,40.8,media,2024-01-04,NaN


Tipos de datos originales:
target_id                  str
str_bool                   str
str_numeric                str
category_col               str
event_date      datetime64[us]
numeric_nan            float64
dtype: object


### Caso 1: Estrategia por defecto (`categorical_strategy='category'`, `bool_as_int=True`)

Configuración recomendada para algoritmos basados en árboles como LightGBM o CatBoost.

In [3]:
df_ml_category = prepare_dataframe_for_ml(
    df=df_raw_ml,
    fillna_cols=["numeric_nan"],
    excluded_cols=["target_id"],
    bool_as_int=True,
    categorical_strategy="category"
)

print("DataFrame preparado (con pandas category y booleanos a 0/1):")
display(df_ml_category)
print("\nTipos resultantes:")
print(df_ml_category.dtypes)

DataFrame preparado (con pandas category y booleanos a 0/1):


,target_id,str_bool,str_numeric,category_col,event_date,numeric_nan
0,ID_1,1,10.5,baja,1704067200000000,100.0
1,ID_2,0,20.0,media,1704153600000000,0.0
2,ID_3,1,30.2,alta,1704240000000000,300.0
3,ID_4,0,40.8,media,1704326400000000,0.0



Tipos resultantes:
target_id            str
str_bool           int64
str_numeric      float64
category_col    category
event_date         int64
numeric_nan      float64
dtype: object


### Caso 2: Estrategia One-Hot Encoding (`categorical_strategy='onehot'`)

Configuración habitual para modelos lineales, SVMs o redes neuronales.

In [4]:
df_ml_onehot = prepare_dataframe_for_ml(
    df=df_raw_ml,
    excluded_cols=["target_id"],
    bool_as_int=True,
    categorical_strategy="onehot"
)

print("DataFrame con One-Hot Encoding:")
display(df_ml_onehot)

DataFrame con One-Hot Encoding:


,target_id,str_bool,str_numeric,event_date,numeric_nan,category_col_alta,category_col_baja,category_col_media
0,ID_1,1,10.5,1704067200000000,100.0,False,True,False
1,ID_2,0,20.0,1704153600000000,NaN,False,False,True
2,ID_3,1,30.2,1704240000000000,300.0,True,False,False
3,ID_4,0,40.8,1704326400000000,NaN,False,False,True


### Caso 3: Estrategia Label Encoding (`categorical_strategy='label'`)

Codificación entera continua de categorías.

In [5]:
df_ml_label = prepare_dataframe_for_ml(
    df=df_raw_ml,
    excluded_cols=["target_id"],
    bool_as_int=True,
    categorical_strategy="label"
)

print("DataFrame con Label Encoding:")
display(df_ml_label)

DataFrame con Label Encoding:


,target_id,str_bool,str_numeric,category_col,event_date,numeric_nan
0,ID_1,1,10.5,1,1704067200000000,100.0
1,ID_2,0,20.0,2,1704153600000000,NaN
2,ID_3,1,30.2,0,1704240000000000,300.0
3,ID_4,0,40.8,2,1704326400000000,NaN
